In [25]:
import sys
import os

# Add project root (parent directory of 'notebooks') to Python path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

print("Project root added:", project_root)
print("Python can now import from src/")


Project root added: /Users/shubhra/hdfcbank-strike-squad
Python can now import from src/


In [26]:
"""
02_data_processing_and_etl.ipynb
Purpose:
    - Clean and preprocess stock data
    - Clean and preprocess synthetic option chain
    - Build ATM selection framework
    - Align risk-free rates
    - Prepare dividend adjustments
"""

import pandas as pd
import numpy as np
from pathlib import Path
import yaml

# Display settings
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 20)

print("Notebook 02: Data Processing & ETL - Environment ready.")


Notebook 02: Data Processing & ETL - Environment ready.


In [27]:
# Load Configuration File

from pathlib import Path
import yaml

# List of possible locations for config.yaml (flexible + robust)
candidate_paths = [
    Path("../config/config.yaml"),
    Path("./config/config.yaml"),
    Path("../config.yaml"),
    Path("./config.yaml")
]

config_path = None
for path in candidate_paths:
    if path.exists():
        config_path = path
        break

if config_path is None:
    raise FileNotFoundError(
        "Unable to locate config.yaml. Expected one of:\n"
        + "\n".join(str(p) for p in candidate_paths)
    )

print(f"Configuration file located at: {config_path}")

# Load YAML configuration
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

print("Configuration loaded successfully.")
print(f"Project Name: {config['project']['name']}")
print(f"Team: {config['project']['team']}\n")


Configuration file located at: ../config/config.yaml
Configuration loaded successfully.
Project Name: HDFCBANK Options Analysis
Team: Strike Squad



In [28]:
# Load all processed datasets created in Notebook 1

import pandas as pd
from pathlib import Path

base = Path("../data/processed")

# ---- Load processed stock data ----
stock_file = base / "hdfc_stock_processed_271P.csv"
if stock_file.exists():
    stock_data = pd.read_csv(stock_file, parse_dates=["Date"])
else:
    stock_data = None
    print("⚠️ Stock processed file missing.")

# ---- Load risk-free data ----
rf_file = base / "risk_free_rates_271P.csv"
if rf_file.exists():
    risk_free_data = pd.read_csv(rf_file, parse_dates=["date"])
else:
    risk_free_data = None
    print("⚠️ Risk-free processed file missing.")

# ---- Load dividend data ----
div_file = base / "hdfc_dividends_271P.csv"
if div_file.exists():
    dividend_data = pd.read_csv(div_file, parse_dates=["ex_date"])
else:
    dividend_data = None
    print("⚠️ Dividend processed file missing.")

# ---- Option chain (raw or processed) ----
opt_raw_path = Path("../data/raw/hdfc_option_chain_raw.csv")
if opt_raw_path.exists():
    option_raw = pd.read_csv(opt_raw_path, parse_dates=["date", "expiry"])
else:
    option_raw = None
    print("⚠️ Option chain raw file missing.")


In [29]:
from pathlib import Path

# Define file paths for raw data
stock_file = Path("../data/raw/hdfc_stock_raw.csv")
risk_free_file = Path("../data/raw/risk_free_raw.csv")
dividend_file = Path("../data/raw/hdfc_dividends_raw.csv")

print("File paths defined:")
print(f"Stock file      : {stock_file}")
print(f"Risk-free file  : {risk_free_file}")
print(f"Dividend file   : {dividend_file}")


File paths defined:
Stock file      : ../data/raw/hdfc_stock_raw.csv
Risk-free file  : ../data/raw/risk_free_raw.csv
Dividend file   : ../data/raw/hdfc_dividends_raw.csv


In [30]:
# =========================================================
# LOAD & FIX STOCK DATA
# =========================================================

stock_file = Path("../data/processed/hdfc_stock_processed_271P.csv")

if not stock_file.exists():
    raise FileNotFoundError(f"Processed stock file not found: {stock_file}")

stock_data = pd.read_csv(stock_file)

# ---- Normalize column names ----
stock_data.columns = [c.lower().strip().replace(" ", "_") for c in stock_data.columns]

# Expected final columns:
# ['date', 'open', 'high', 'low', 'close', 'volume', 'adj_close']

# ---- Convert date column ----
if "date" not in stock_data.columns:
    raise KeyError("The processed stock file is missing a 'Date' column.")

stock_data["date"] = pd.to_datetime(stock_data["date"], errors="coerce")

# ---- Sort by date ----
stock_data = stock_data.sort_values("date").reset_index(drop=True)

# ---- Compute log returns (if missing) ----
if "log_return" not in stock_data.columns:
    stock_data["log_return"] = np.log(stock_data["adj_close"] / stock_data["adj_close"].shift(1))

# ---- Compute rolling 30D realized volatility ----
if "realized_vol_annual" not in stock_data.columns:
    stock_data["realized_vol_30d"] = (
        stock_data["log_return"].rolling(30).std()
    )

    stock_data["realized_vol_annual"] = (
        stock_data["realized_vol_30d"] * np.sqrt(252)
    )

# ---- Drop initial rows with NaNs from rolling window ----
stock_data = stock_data.dropna(subset=["realized_vol_annual"])

# ---- Summary ----
print("\nCleaned & enriched stock data loaded successfully.\n")
print(f"Total trading days       : {len(stock_data)}")
print(f"Date range               : {stock_data['date'].min().date()} to {stock_data['date'].max().date()}")
print(f"Mean annualized volatility: {stock_data['realized_vol_annual'].mean():.2%}")



Cleaned & enriched stock data loaded successfully.

Total trading days       : 478
Date range               : 2023-12-15 to 2025-11-20
Mean annualized volatility: 18.79%


In [31]:
# CELL 5: Load and Validate Raw HDFCBANK Stock Data

from pathlib import Path
import pandas as pd

# Path to your raw stock data
stock_file = Path("../data/raw/hdfc_stock_raw.csv")

print(f"Looking for stock file at: {stock_file}")

# --- File existence check ---
if not stock_file.exists():
    raise FileNotFoundError(f"Raw stock data not found at: {stock_file}")

# --- Load CSV ---
stock_raw = pd.read_csv(stock_file)

print("Raw stock data loaded successfully.")
print(f"Rows loaded: {len(stock_raw):,}\n")

# --- Required columns based on YOUR actual CSV structure ---
required_cols = ["Date", "Open", "High", "Low", "Close", "Volume"]

missing_cols = [c for c in required_cols if c not in stock_raw.columns]

if missing_cols:
    raise ValueError(f"Missing columns in stock data: {missing_cols}")

# --- Create Adj Close column if missing ---
if "Adj Close" not in stock_raw.columns:
    print("Column 'Adj Close' not found. Creating it using 'Close'...")
    stock_raw["Adj Close"] = stock_raw["Close"]

# --- Convert date column ---
stock_raw["Date"] = pd.to_datetime(stock_raw["Date"], errors="coerce")

# Drop rows where date failed to parse
stock_raw = stock_raw.dropna(subset=["Date"]).copy()

# Sort by date
stock_raw = stock_raw.sort_values("Date")

print("Stock data validated and cleaned.\n")

# --- Quick summary ---
print("Summary:")
print(f"Date range : {stock_raw['Date'].min().date()} → {stock_raw['Date'].max().date()}")
print(f"Total rows : {len(stock_raw):,}")
print(f"Columns    : {list(stock_raw.columns)}")

# Save cleaned version in memory for next cells
stock_data = stock_raw.copy()


Looking for stock file at: ../data/raw/hdfc_stock_raw.csv
Raw stock data loaded successfully.
Rows loaded: 508

Column 'Adj Close' not found. Creating it using 'Close'...
Stock data validated and cleaned.

Summary:
Date range : 2023-11-01 → 2025-11-20
Total rows : 508
Columns    : ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Adj Close']


In [32]:
# --- Add Log Returns ---
stock_data['Log_Return'] = np.log(
    stock_data['Adj Close'] / stock_data['Adj Close'].shift(1)
)

# --- 30-Day Rolling Realized Volatility ---
stock_data['Realized_Vol_30D'] = (
    stock_data['Log_Return']
    .rolling(window=30)
    .std()
)

# --- Annualize Volatility ---
stock_data['Realized_Vol_Annual'] = (
    stock_data['Realized_Vol_30D'] * np.sqrt(252)
)

# Clean up any initial NaNs
stock_data = stock_data.dropna(subset=['Realized_Vol_Annual']).reset_index(drop=True)


In [33]:
from pathlib import Path

# Path to raw option chain data (synthetic or real)
option_chain_file = Path("../data/raw/hdfc_option_chain_raw.csv")

print("Option chain path set to:", option_chain_file)

Option chain path set to: ../data/raw/hdfc_option_chain_raw.csv


In [34]:
# Load raw option chain data (synthetic or real)

if not option_chain_file.exists():
    raise FileNotFoundError(
        f"Option chain file not found at: {option_chain_file}\n"
        "Ensure you have placed your option chain CSV in data/raw/"
    )

option_raw = pd.read_csv(option_chain_file)

print("Option chain data loaded.")
print(f"Total rows: {len(option_raw)}\n")

print("Columns in option chain:")
print(option_raw.columns.tolist())

# Convert dates if present
date_cols = ["date", "Date", "expiry", "Expiry"]
for col in date_cols:
    if col in option_raw.columns:
        option_raw[col] = pd.to_datetime(option_raw[col], errors="coerce")

# Sort by date if possible
if "date" in option_raw.columns:
    option_raw = option_raw.sort_values("date").reset_index(drop=True)

# Preview
display(option_raw.head(10))
display(option_raw.tail(10))


Option chain data loaded.
Total rows: 8694

Columns in option chain:
['date', 'expiry', 'strike', 'days_to_expiry', 'stock_price', 'call_bid', 'call_ask', 'call_mid', 'put_bid', 'put_ask', 'put_mid', 'implied_vol', 'open_interest', 'volume', 'risk_free_rate', 'moneyness', 'call_spread_pct', 'put_spread_pct']


,date,expiry,strike,days_to_expiry,stock_price,call_bid,call_ask,call_mid,put_bid,put_ask,put_mid,implied_vol,open_interest,volume,risk_free_rate,moneyness,call_spread_pct,put_spread_pct
0,2023-12-15,2024-01-18,720,34,806.219299,90.35,91.25,90.781899,0.05,0.00,0.016356,0.134515,4533,247,0.068,1.119749,0.991387,-305.701209
1,2023-12-15,2024-01-18,880,34,806.219299,0.30,0.30,0.309828,68.20,68.90,68.534008,0.134515,4332,230,0.068,0.916158,0.000000,1.021391
2,2023-12-15,2024-01-18,870,34,806.219299,0.60,0.65,0.626333,58.60,59.20,58.913655,0.134515,5379,246,0.068,0.926689,7.982978,1.018440
3,2023-12-15,2024-01-18,860,34,806.219299,1.20,1.20,1.198918,49.20,49.90,49.549383,0.134515,4399,293,0.068,0.937464,0.000000,1.412732
4,2023-12-15,2024-01-18,850,34,806.219299,2.15,2.20,2.173397,40.30,40.90,40.587005,0.134515,5734,362,0.068,0.948493,2.300546,1.478306
5,2023-12-15,2024-01-18,830,34,806.219299,6.00,6.15,6.082368,24.45,24.80,24.622259,0.134515,4438,206,0.068,0.971349,2.466145,1.421478
6,2023-12-15,2024-01-18,820,34,806.219299,9.25,9.55,9.413760,17.90,18.15,18.016794,0.134515,4191,321,0.068,0.983194,3.186824,1.387594
7,2023-12-15,2024-01-18,810,34,806.219299,13.75,13.95,13.870425,12.45,12.65,12.536601,0.134515,5948,150,0.068,0.995332,1.441917,1.595329
8,2023-12-15,2024-01-18,840,34,806.219299,3.70,3.80,3.733258,31.95,32.45,32.210008,0.134515,5554,346,0.068,0.959785,2.678625,1.552313
9,2023-12-15,2024-01-18,790,34,806.219299,26.10,26.50,26.287392,5.00,5.15,5.079853,0.134515,5626,192,0.068,1.020531,1.521642,2.952841


,date,expiry,strike,days_to_expiry,stock_price,call_bid,call_ask,call_mid,put_bid,put_ask,put_mid,implied_vol,open_interest,volume,risk_free_rate,moneyness,call_spread_pct,put_spread_pct
8684,2025-11-20,2025-12-25,970,35,1008.849976,45.95,46.65,46.286634,1.20,1.25,1.224716,0.106372,4047,388,0.067,1.040052,1.512316,4.082578
8685,2025-11-20,2025-12-25,960,35,1008.849976,55.30,55.85,55.586077,0.60,0.60,0.588201,0.106372,4396,272,0.067,1.050885,0.989456,0.000000
8686,2025-11-20,2025-12-25,950,35,1008.849976,64.85,65.50,65.192623,0.25,0.25,0.258787,0.106372,5683,547,0.067,1.061947,0.997045,0.000000
8687,2025-11-20,2025-12-25,940,35,1008.849976,74.60,75.35,74.973535,0.10,0.10,0.103740,0.106372,5303,238,0.067,1.073245,1.000353,0.000000
8688,2025-11-20,2025-12-25,930,35,1008.849976,84.40,85.25,84.843448,0.05,0.05,0.037693,0.106372,5731,354,0.067,1.084785,1.001845,0.000000
8689,2025-11-20,2025-12-25,920,35,1008.849976,94.30,95.25,94.754063,0.05,0.00,0.012349,0.106372,4180,403,0.067,1.096576,1.002596,-404.880507
8690,2025-11-20,2025-12-25,910,35,1008.849976,104.15,105.20,104.681303,0.05,0.00,0.003630,0.106372,4845,353,0.067,1.108626,1.003044,-1377.475557
8691,2025-11-20,2025-12-25,900,35,1008.849976,114.05,115.20,114.614585,0.05,0.00,0.000952,0.106372,3882,344,0.067,1.120944,1.003363,-5250.229618
8692,2025-11-20,2025-12-25,990,35,1008.849976,29.15,29.60,29.363918,4.10,4.25,4.173920,0.106372,5499,216,0.067,1.019040,1.532493,3.593744
8693,2025-11-20,2025-12-25,1100,35,1008.849976,0.10,0.10,0.085666,83.75,84.60,84.191221,0.106372,4393,208,0.067,0.917136,0.000000,1.009606


In [35]:
# Clean and process option chain data

option_df = option_raw.copy()

# Normalize column names to lowercase
option_df.columns = [c.strip().lower().replace(" ", "_") for c in option_df.columns]

required_cols = ["date", "strike", "expiry", "call_bid", "call_ask", "put_bid", "put_ask"]
missing_cols = [c for c in required_cols if c not in option_df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns in option chain: {missing_cols}")

# Drop rows without dates or strikes
option_df = option_df.dropna(subset=["date", "strike"]).copy()

# Compute mid prices
option_df["call_mid"] = (option_df["call_bid"] + option_df["call_ask"]) / 2
option_df["put_mid"]  = (option_df["put_bid"]  + option_df["put_ask"])  / 2

# Compute bid-ask spread % relative to mid price
option_df["spread_call_pct"] = np.where(
    option_df["call_mid"] > 0,
    (option_df["call_ask"] - option_df["call_bid"]) / option_df["call_mid"] * 100,
    np.nan
)
option_df["spread_put_pct"] = np.where(
    option_df["put_mid"] > 0,
    (option_df["put_ask"] - option_df["put_bid"]) / option_df["put_mid"] * 100,
    np.nan
)

# Merge with config for liquidity filters
liq = config["liquidity"]
min_oi = liq.get("min_open_interest", 100)
max_spread = liq.get("max_bid_ask_spread_pct", 5.0)
min_price = liq.get("min_option_premium", 5)

# Add liquidity flag
option_df["liquid"] = (
    (option_df.get("open_interest", np.inf) >= min_oi) &
    (option_df["call_mid"] >= min_price) &
    (option_df["put_mid"]  >= min_price) &
    (option_df["spread_call_pct"] <= max_spread) &
    (option_df["spread_put_pct"]  <= max_spread)
)

print("Option chain processed.")
print(f"Total rows after cleaning: {len(option_df)}")
print(f"Liquid contracts: {option_df['liquid'].sum()}")

display(option_df.head(15))


Option chain processed.
Total rows after cleaning: 8694
Liquid contracts: 4022


,date,expiry,strike,days_to_expiry,stock_price,call_bid,call_ask,call_mid,put_bid,put_ask,...,implied_vol,open_interest,volume,risk_free_rate,moneyness,call_spread_pct,put_spread_pct,spread_call_pct,spread_put_pct,liquid
0,2023-12-15,2024-01-18,720,34,806.219299,90.35,91.25,90.800,0.05,0.00,...,0.134515,4533,247,0.068,1.119749,0.991387,-305.701209,0.991189,-200.000000,False
1,2023-12-15,2024-01-18,880,34,806.219299,0.30,0.30,0.300,68.20,68.90,...,0.134515,4332,230,0.068,0.916158,0.000000,1.021391,0.000000,1.021152,False
2,2023-12-15,2024-01-18,870,34,806.219299,0.60,0.65,0.625,58.60,59.20,...,0.134515,5379,246,0.068,0.926689,7.982978,1.018440,8.000000,1.018676,False
3,2023-12-15,2024-01-18,860,34,806.219299,1.20,1.20,1.200,49.20,49.90,...,0.134515,4399,293,0.068,0.937464,0.000000,1.412732,0.000000,1.412714,False
4,2023-12-15,2024-01-18,850,34,806.219299,2.15,2.20,2.175,40.30,40.90,...,0.134515,5734,362,0.068,0.948493,2.300546,1.478306,2.298851,1.477833,False
5,2023-12-15,2024-01-18,830,34,806.219299,6.00,6.15,6.075,24.45,24.80,...,0.134515,4438,206,0.068,0.971349,2.466145,1.421478,2.469136,1.421320,True
6,2023-12-15,2024-01-18,820,34,806.219299,9.25,9.55,9.400,17.90,18.15,...,0.134515,4191,321,0.068,0.983194,3.186824,1.387594,3.191489,1.386963,True
7,2023-12-15,2024-01-18,810,34,806.219299,13.75,13.95,13.850,12.45,12.65,...,0.134515,5948,150,0.068,0.995332,1.441917,1.595329,1.444043,1.593625,True
8,2023-12-15,2024-01-18,840,34,806.219299,3.70,3.80,3.750,31.95,32.45,...,0.134515,5554,346,0.068,0.959785,2.678625,1.552313,2.666667,1.552795,False
9,2023-12-15,2024-01-18,790,34,806.219299,26.10,26.50,26.300,5.00,5.15,...,0.134515,5626,192,0.068,1.020531,1.521642,2.952841,1.520913,2.955665,True


In [36]:
# Load risk-free rate data (RBI 91-Day T-Bill)

if not risk_free_file.exists():
    raise FileNotFoundError(
        f"Risk-free rate file not found at: {risk_free_file}\n"
        "Place your RBI 91-day T-Bill CSV in data/reference/"
    )

rf_raw = pd.read_csv(risk_free_file)

print("Risk-free rate data loaded.")
print(f"Total rows: {len(rf_raw)}\n")

print("Columns in risk-free rate file:")
print(rf_raw.columns.tolist())

# Convert date column if present
date_cols = ["date", "Date"]
for col in date_cols:
    if col in rf_raw.columns:
        rf_raw[col] = pd.to_datetime(rf_raw[col], errors="coerce")

# Sort by date
if "date" in rf_raw.columns:
    rf_raw = rf_raw.sort_values("date").reset_index(drop=True)

display(rf_raw.head(10))
display(rf_raw.tail(10))


Risk-free rate data loaded.
Total rows: 1055

Columns in risk-free rate file:
['date', 'rate']


,date,rate
0,2023-01-02,0.064500
1,2023-01-03,0.064502
2,2023-01-04,0.064503
3,2023-01-05,0.064505
4,2023-01-06,0.064506
5,2023-01-07,0.064507
6,2023-01-08,0.064509
7,2023-01-09,0.064510
8,2023-01-10,0.064512
9,2023-01-11,0.064514


,date,rate
1045,2025-11-12,0.067315
1046,2025-11-13,0.067316
1047,2025-11-14,0.067317
1048,2025-11-15,0.067318
1049,2025-11-16,0.067319
1050,2025-11-17,0.067320
1051,2025-11-18,0.067321
1052,2025-11-19,0.067322
1053,2025-11-20,0.067323
1054,2025-11-21,0.067324


In [37]:
# Load dividend data for HDFCBANK

if not dividend_file.exists():
    raise FileNotFoundError(
        f"Dividend data file not found at: {dividend_file}\n"
        "Place your HDFCBANK dividend CSV in data/reference/"
    )

div_raw = pd.read_csv(dividend_file)

print("Dividend data loaded.")
print(f"Total rows: {len(div_raw)}\n")

print("Columns in dividend file:")
print(div_raw.columns.tolist())

# Convert date column
date_cols = ["ex_date", "ExDate", "date", "Date"]
for col in date_cols:
    if col in div_raw.columns:
        div_raw[col] = pd.to_datetime(div_raw[col], errors="coerce")
        break  # use first matching date column

# Sort by ex-dividend date
for col in date_cols:
    if col in div_raw.columns:
        div_raw = div_raw.sort_values(col).reset_index(drop=True)
        break

display(div_raw.head(10))
display(div_raw.tail(10))


Dividend data loaded.
Total rows: 5

Columns in dividend file:
['Ex_Date', 'Dividend_Amount']


,Ex_Date,Dividend_Amount
0,2022-05-12 00:00:00+05:30,7.75
1,2023-05-16 00:00:00+05:30,9.50
2,2024-05-10 00:00:00+05:30,9.75
3,2025-06-27 00:00:00+05:30,11.00
4,2025-07-25 00:00:00+05:30,2.50


,Ex_Date,Dividend_Amount
0,2022-05-12 00:00:00+05:30,7.75
1,2023-05-16 00:00:00+05:30,9.50
2,2024-05-10 00:00:00+05:30,9.75
3,2025-06-27 00:00:00+05:30,11.00
4,2025-07-25 00:00:00+05:30,2.50


In [38]:
# Load processed risk-free rates
from pathlib import Path
import pandas as pd

risk_free_processed_file = Path("../data/processed/risk_free_rates_271P.csv")

if not risk_free_processed_file.exists():
    raise FileNotFoundError(
        f"Processed risk-free rate file not found at: {risk_free_processed_file}\n"
        "Run Notebook 1 to generate it."
    )

risk_free_data = pd.read_csv(risk_free_processed_file)

print("Processed risk-free rate data loaded successfully.")
print(f"Total rows: {len(risk_free_data)}")
display(risk_free_data.head())


Processed risk-free rate data loaded successfully.
Total rows: 752


,date,rate_annual
0,2023-11-01,0.068
1,2023-11-02,0.068
2,2023-11-03,0.068
3,2023-11-04,0.068
4,2023-11-05,0.068


In [39]:
# ==== Compute Log Returns & Realized Volatility for Stock Data ====

import numpy as np
import pandas as pd

df = stock_data.copy()

# Ensure Date is datetime
df["Date"] = pd.to_datetime(df["Date"])

# Sort by Date just in case
df = df.sort_values("Date").reset_index(drop=True)

# ---- 1. Log Returns ----
df["Log_Return"] = np.log(df["Adj Close"] / df["Adj Close"].shift(1))

# ---- 2. 30-Day Rolling Volatility (daily) ----
# Calculate rolling standard deviation of daily log returns
df["Vol_30D"] = df["Log_Return"].rolling(window=30).std()

# ---- 3. Annualized Volatility ----
TRADING_DAYS = 252   # Standard annualization factor
df["Vol_Annual"] = df["Vol_30D"] * np.sqrt(TRADING_DAYS)

# ---- Clean up initial rows ----
df = df.dropna().reset_index(drop=True)

# Replace your in-memory stock_data with processed version
stock_data = df

print("Volatility columns successfully added.")
print(stock_data.head())
print(stock_data.tail())


Volatility columns successfully added.
                       Date        Open        High         Low       Close  \
0 2024-01-31 00:00:00+05:30  700.584116  717.837102  699.270088  711.802246   
1 2024-02-01 00:00:00+05:30  713.018956  717.180083  708.638784  713.651611   
2 2024-02-02 00:00:00+05:30  717.861428  720.708527  701.825157  703.820557   
3 2024-02-05 00:00:00+05:30  703.747506  706.618963  697.931635  703.187805   
4 2024-02-06 00:00:00+05:30  703.528606  705.548304  697.225990  702.822876   

     Volume   Adj Close  Log_Return  Realized_Vol_30D  Realized_Vol_Annual  \
0  65761040  711.802246    0.012557          0.020398             0.323813   
1  32690740  713.651611    0.002595          0.020425             0.324241   
2  44867754  703.820557   -0.013871          0.020498             0.325392   
3  38605046  703.187805   -0.000899          0.020468             0.324914   
4  41075740  702.822876   -0.000519          0.020048             0.318246   

    Vol_30D  Vol_

In [40]:
# Load dividend data if available

dividend_file = Path("../data/raw/hdfc_dividends_raw.csv")

if dividend_file.exists():
    try:
        dividend_data = pd.read_csv(dividend_file)
        print("Dividend data loaded.")
        print(f"Rows: {len(dividend_data)}")
        
        # Convert date
        if "Ex_Date" in dividend_data.columns:
            dividend_data["Ex_Date"] = pd.to_datetime(dividend_data["Ex_Date"], errors="coerce")

    except Exception as e:
        print("Error loading dividend file.")
        print("Error:", e)
        dividend_data = pd.DataFrame()
else:
    print("Dividend file not found. Proceeding without dividend data.")
    dividend_data = pd.DataFrame()


Dividend data loaded.
Rows: 5


In [41]:
# Clean Stock Data + Compute Volatility

import pandas as pd
import numpy as np
from pathlib import Path

# ----------- Load Raw Stock Data -----------
stock_file = Path("../data/raw/hdfc_stock_raw.csv")

if not stock_file.exists():
    raise FileNotFoundError(f"Raw stock data not found at: {stock_file}")

stock_raw = pd.read_csv(stock_file)

# ----------- Standardize & Clean Columns -----------
stock_raw.columns = [c.strip().lower() for c in stock_raw.columns]

# Expected columns for your case (OHLCV only)
required_cols = ["date", "open", "high", "low", "close", "volume"]
missing = [c for c in required_cols if c not in stock_raw.columns]
if missing:
    raise ValueError(f"Missing required columns in stock CSV: {missing}")

# Convert date column
stock_raw["date"] = pd.to_datetime(stock_raw["date"], errors="coerce")

# Drop invalid dates
stock_raw = stock_raw.dropna(subset=["date"]).copy()

# Sort chronologically
stock_raw = stock_raw.sort_values("date").reset_index(drop=True)

# ----------- Compute Log Returns (using Close) -----------
stock_raw["log_return"] = np.log(stock_raw["close"] / stock_raw["close"].shift(1))

# ----------- Rolling Realized Volatility (30-Day Window) -----------
stock_raw["realized_vol_30d"] = stock_raw["log_return"].rolling(window=30).std()

# Annualize
stock_raw["realized_vol_annual"] = stock_raw["realized_vol_30d"] * np.sqrt(252)

# Drop initial NaN window
stock_data = stock_raw.dropna(subset=["realized_vol_annual"]).copy()

# ----------- Summary -----------

print("Cleaned stock data loaded successfully.\n")
print(f"Total trading days       : {len(stock_data):,}")
print(f"Date range               : {stock_data['date'].min().date()} to {stock_data['date'].max().date()}")
print(f"Mean annualized volatility: {stock_data['realized_vol_annual'].mean():.2%}")

print("\nPreview:")
display(stock_data.head())


Cleaned stock data loaded successfully.

Total trading days       : 478
Date range               : 2023-12-15 to 2025-11-20
Mean annualized volatility: 18.79%

Preview:


,date,open,high,low,close,volume,log_return,realized_vol_30d,realized_vol_annual
30,2023-12-15 00:00:00+05:30,803.031484,811.791827,801.376740,806.219299,138837130,0.003871,0.008474,0.134515
31,2023-12-18 00:00:00+05:30,808.238971,810.185714,803.493785,805.805542,18014754,-0.000513,0.008501,0.134954
32,2023-12-19 00:00:00+05:30,803.031471,807.241314,800.135714,804.442871,24008446,-0.001692,0.008558,0.135855
33,2023-12-20 00:00:00+05:30,806.924918,812.254103,800.330303,806.438232,34232520,0.002477,0.008533,0.135452
34,2023-12-21 00:00:00+05:30,806.438279,822.377238,803.153151,820.892822,36589240,0.017765,0.008770,0.139222


In [42]:
# CELL: Load all processed datasets created in Notebook 1

import pandas as pd
from pathlib import Path

base = Path("../data/processed")

# ---- Load processed stock data ----
stock_file = base / "hdfc_stock_processed_271P.csv"
if stock_file.exists():
    stock_data = pd.read_csv(stock_file, parse_dates=["Date"])
else:
    stock_data = None
    print("⚠️ Stock processed file missing.")

# ---- Load risk-free data ----
rf_file = base / "risk_free_rates_271P.csv"
if rf_file.exists():
    risk_free_data = pd.read_csv(rf_file, parse_dates=["date"])
else:
    risk_free_data = None
    print("⚠️ Risk-free processed file missing.")

# ---- Load dividend data ----
div_file = base / "hdfc_dividends_271P.csv"
if div_file.exists():
    dividend_data = pd.read_csv(div_file, parse_dates=["ex_date"])
else:
    dividend_data = None
    print("⚠️ Dividend processed file missing.")

# ---- Option chain (raw or processed) ----
opt_raw_path = Path("../data/raw/hdfc_option_chain_raw.csv")
if opt_raw_path.exists():
    option_raw = pd.read_csv(opt_raw_path, parse_dates=["date", "expiry"])
else:
    option_raw = None
    print("⚠️ Option chain raw file missing.")


In [43]:
# DIAGNOSTIC + FALLBACK: find & load option chain OR generate synthetic one
import pandas as pd
import numpy as np
from pathlib import Path
import math
from datetime import timedelta

# candidate folders to search (relative to notebook)
candidates = [
    Path("../data/processed"),
    Path("../data/raw"),
    Path("../data"),
    Path("."),
    Path("../")
]

# candidate filenames (common names used in this project)
fnames = [
    "hdfc_option_chain_processed.csv",
    "hdfc_option_chain_raw.csv",
    "hdfc_option_chain.csv",
    "option_chain_hdfc.csv",
    "hdfc_options.csv",
    "hdfc_option_chain_processed_271P.csv"
]

found = None
for folder in candidates:
    if not folder.exists():
        continue
    for f in fnames:
        p = folder / f
        if p.exists():
            found = p
            break
    if found:
        break

print("Searched folders:", [str(p) for p in candidates])
if found:
    print("Found option-chain file at:", found.resolve())
    try:
        opt = pd.read_csv(found)
        print("Loaded option chain: rows =", len(opt))
        print("Columns:", opt.columns.tolist())
        display(opt.head(5))
    except Exception as e:
        print("Error while loading the found file:", e)
else:
    print("No existing option-chain file found in expected locations.")
    print()
    print("FALLBACK: I'll now attempt to generate a synthetic option chain using your enriched stock file.")
    # paths we will try to use for inputs
    stock_enriched = Path("../data/processed/hdfc_stock_processed_271P_enriched.csv")
    rf_aligned = Path("../data/processed/risk_free_rates_271P.csv")
    out_path = Path("../data/processed/hdfc_option_chain_processed.csv")
    # check inputs
    if not stock_enriched.exists():
        print("Missing stock enriched file:", stock_enriched)
        print(" -> If you have a different processed stock filename, update the path or place the enriched file at this location.")
    if not rf_aligned.exists():
        print("Missing risk-free file:", rf_aligned)
        print(" -> If missing, we'll assume a flat fallback r = 0.065 (6.5%).")
    if not stock_enriched.exists():
        raise FileNotFoundError("Cannot generate synthetic option chain: enriched stock file not found.")
    # load inputs
    stock_df = pd.read_csv(stock_enriched, parse_dates=["date"])
    if rf_aligned.exists():
        rf_df = pd.read_csv(rf_aligned, parse_dates=["date"])
    else:
        rf_df = None

    # Basic synthetic generator (ATM only, one expiry ~30 days ahead, strikes +/- 20% grid)
    print("\nGenerating synthetic option chain (basic ATM-centered) — this may take a minute...\n")

    # helper: Black-Scholes price (European) - uses normal CDF via scipy if available, else numpy approx
    try:
        from scipy.stats import norm
        def norm_cdf(x): return norm.cdf(x)
    except Exception:
        # fallback simple approximation (not ideal but works)
        def norm_cdf(x): return 0.5*(1 + np.erf(x/np.sqrt(2)))

    def bsm_price(S, K, T, r, sigma, option_type="call", q=0.0):
        # S, K float; T in years; r,q continuous rates; sigma annual vol
        if T <= 0:
            if option_type=="call":
                return max(S-K, 0.0)
            return max(K-S, 0.0)
        if sigma <= 0:
            # zero vol -> discounted intrinsic
            if option_type=="call":
                return max(S * math.exp(-q*T) - K*math.exp(-r*T), 0.0)
            else:
                return max(K*math.exp(-r*T) - S*math.exp(-q*T), 0.0)
        d1 = (math.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
        d2 = d1 - sigma * math.sqrt(T)
        if option_type == "call":
            return S * math.exp(-q*T) * norm_cdf(d1) - K * math.exp(-r*T) * norm_cdf(d2)
        else:
            return K * math.exp(-r*T) * norm_cdf(-d2) - S * math.exp(-q*T) * norm_cdf(-d1)

    # prepare strikes grid function
    def make_strike_grid(S0, step=10, pct_range=0.2):
        base = round(S0 / step) * step
        low = int(base * (1 - pct_range))
        high = int(base * (1 + pct_range))
        strikes = list(range(max(step, low), high + step, step))
        # ensure S0 near center
        if base not in strikes:
            strikes.append(base)
            strikes = sorted(list(set(strikes)))
        return strikes

    rows = []
    for idx, row in stock_df.iterrows():
        S0 = float(row.get("adj_close") if "adj_close" in row else row.get("close"))
        trade_date = pd.to_datetime(row["date"]).date()
        # choose expiry ~30 calendar days ahead (find next trading date in stock_df close to +30 days)
        candidate_expiry = trade_date + pd.Timedelta(days=30)
        # find the closest available date in stock_df
        # convert stock_df.date to date-only Series for lookup
        dates = pd.to_datetime(stock_df["date"]).dt.date
        if candidate_expiry in set(dates):
            expiry = candidate_expiry
        else:
            # pick the trading date in stock_df that is closest after candidate_expiry
            future_dates = [d for d in dates if d >= candidate_expiry]
            if len(future_dates) == 0:
                expiry = dates.iloc[-1]
            else:
                expiry = min(future_dates)
        # days to expiry
        dte = (pd.to_datetime(expiry) - pd.to_datetime(trade_date)).days
        T = max(dte / 365.0, 1/365.0)

        # risk-free for this date (use rf_df if available, else fallback)
        if rf_df is not None and trade_date in set(pd.to_datetime(rf_df["date"]).dt.date):
            r_row = rf_df[pd.to_datetime(rf_df["date"]).dt.date == trade_date]
            r = float(r_row.iloc[0].get("rate_annual", r_row.iloc[0].iloc[-1]))
        else:
            r = 0.065

        # rough historical vol: use realized_vol_30d if present, else compute from recent returns
        if "realized_vol_annual" in stock_df.columns:
            sigma = float(row.get("realized_vol_annual")) if not np.isnan(row.get("realized_vol_annual")) else 0.20
        else:
            sigma = 0.20

        strikes = make_strike_grid(S0, step=10, pct_range=0.2)
        for K in strikes:
            call_mid = bsm_price(S0, K, T, r, sigma, option_type="call")
            put_mid = bsm_price(S0, K, T, r, sigma, option_type="put")
            # add small synthetic spread and OI/volume placeholders
            spread = max(0.5, max(0.002*call_mid, 0.5))  # min 0.5 rupee
            call_bid = max(0.01, call_mid - spread/2)
            call_ask = call_mid + spread/2
            put_bid = max(0.01, put_mid - spread/2)
            put_ask = put_mid + spread/2
            rows.append({
                "date": pd.to_datetime(trade_date),
                "expiry": pd.to_datetime(expiry),
                "strike": K,
                "days_to_expiry": dte,
                "stock_price": S0,
                "call_bid": round(call_bid, 2),
                "call_ask": round(call_ask, 2),
                "call_mid": round(call_mid, 2),
                "put_bid": round(put_bid, 2),
                "put_ask": round(put_ask, 2),
                "put_mid": round(put_mid, 2),
                "implied_vol": round(sigma, 4),
                "open_interest": 500,
                "volume": 10,
                "risk_free_rate": r,
            })

    opt_df = pd.DataFrame(rows)
    # basic sanity checks
    print("Synthetic option chain rows:", len(opt_df))
    print("Saving synthetic option chain to:", out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    opt_df.to_csv(out_path, index=False)
    print("Saved. Preview:")
    display(opt_df.head(10))
    print("\nNow update your notebook option_chain_file path to point to:")
    print("  ../data/processed/hdfc_option_chain_processed.csv")


Searched folders: ['../data/processed', '../data/raw', '../data', '.', '..']
Found option-chain file at: /Users/shubhra/hdfcbank-strike-squad/data/raw/hdfc_option_chain_raw.csv
Loaded option chain: rows = 8694
Columns: ['date', 'expiry', 'strike', 'days_to_expiry', 'stock_price', 'call_bid', 'call_ask', 'call_mid', 'put_bid', 'put_ask', 'put_mid', 'implied_vol', 'open_interest', 'volume', 'risk_free_rate', 'moneyness', 'call_spread_pct', 'put_spread_pct']


,date,expiry,strike,days_to_expiry,stock_price,call_bid,call_ask,call_mid,put_bid,put_ask,put_mid,implied_vol,open_interest,volume,risk_free_rate,moneyness,call_spread_pct,put_spread_pct
0,2023-12-15,2024-01-18,720,34,806.219299,90.35,91.25,90.781899,0.05,0.00,0.016356,0.134515,4533,247,0.068,1.119749,0.991387,-305.701209
1,2023-12-15,2024-01-18,730,34,806.219299,80.45,81.30,80.878702,0.05,0.05,0.050017,0.134515,3876,360,0.068,1.104410,1.050957,0.000000
2,2023-12-15,2024-01-18,740,34,806.219299,70.65,71.40,71.028832,0.15,0.15,0.137004,0.134515,4376,120,0.068,1.089486,1.055909,0.000000
3,2023-12-15,2024-01-18,750,34,806.219299,61.00,61.60,61.293289,0.35,0.35,0.338319,0.134515,5646,507,0.068,1.074959,0.978900,0.000000
4,2023-12-15,2024-01-18,760,34,806.219299,51.50,52.05,51.776161,0.75,0.75,0.758049,0.134515,5792,457,0.068,1.060815,1.062265,0.000000


In [44]:
# CELL: Project data summary (robust)
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime

# ---- Config: paths to files uploaded in this session ----
STOCK_PATHS = [
    Path("/mnt/data/hdfc_stock_processed_271P.csv"),
    Path("/mnt/data/hdfc_stock_processed_271P_enriched.csv"),
    Path("/mnt/data/hdfc_stock_processed.csv"),
    Path("../data/processed/hdfc_stock_processed_271P.csv")
]
OPTION_CHAIN_PATHS = [
    Path("/mnt/data/hdfc_option_chain_processed_0271.csv"),
    Path("/mnt/data/option_chain_synthetic_0271.csv"),
    Path("../data/processed/hdfc_option_chain_processed_0271.csv"),
    Path("../data/raw/hdfc_option_chain_raw.csv")
]
RISK_FREE_PATHS = [
    Path("/mnt/data/risk_free_rates_271P.csv"),
    Path("../data/processed/risk_free_rates_271P.csv"),
    Path("../data/reference/risk_free_raw.csv")
]
DIVIDEND_PATHS = [
    Path("/mnt/data/hdfc_dividends_271P.csv"),
    Path("../data/processed/hdfc_dividends_271P.csv"),
    Path("../data/reference/hdfc_dividends.csv")
]

def find_first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

def _load_csv(path):
    df = pd.read_csv(path)
    # normalize column names for robust access
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    return df

summary = {}

# ---- Stock data ----
stock_file = find_first_existing(STOCK_PATHS)
if stock_file is None:
    print("Stock Data: NOT FOUND. Expected one of:\n ", "\n  ".join(str(p) for p in STOCK_PATHS))
    summary['stock'] = None
else:
    stock = _load_csv(stock_file)
    # detect date column
    date_cols = [c for c in stock.columns if c in ("date", "datetime", "timestamp")]
    if date_cols:
        stock['date'] = pd.to_datetime(stock[date_cols[0]], errors="coerce")
        stock = stock.sort_values('date').reset_index(drop=True)
    else:
        # try index as date
        stock['date'] = pd.to_datetime(stock.get('index', None), errors="coerce")
    # determine adjusted close column
    adj_candidates = ['adj_close','adjusted_close','adjusted_close_price','adjclose','adj close','adj_close_price']
    adj_col = next((c for c in adj_candidates if c in stock.columns), None)
    close_col = next((c for c in ['close','close_price','last'] if c in stock.columns), None)
    price_col = adj_col or close_col
    # compute log returns and rolling vol if missing
    if price_col and 'log_return' not in stock.columns:
        stock['price'] = pd.to_numeric(stock[price_col], errors='coerce')
        stock['log_return'] = np.log(stock['price'] / stock['price'].shift(1))
    # rolling realized vol (30d) and annualized if present/needed
    vol_col = None
    vol_candidates = ['realized_vol_annual','vol_ann','vol_annual','realized_vol','volatility','vol30','vol_30','vol_30d']
    for c in vol_candidates:
        if c in stock.columns:
            vol_col = c
            break
    if vol_col is None:
        # compute 30-day rolling std -> annualized
        if 'log_return' in stock.columns:
            stock['vol30'] = stock['log_return'].rolling(window=30, min_periods=20).std()
            stock['vol_ann'] = stock['vol30'] * (252**0.5)
            vol_col = 'vol_ann'
    # summary values
    stock_rows = len(stock)
    try:
        date_min = stock['date'].min().date()
        date_max = stock['date'].max().date()
    except Exception:
        date_min = date_max = "N/A"
    avg_vol = None
    if vol_col and vol_col in stock.columns:
        avg_vol = stock[vol_col].dropna().mean()
    print("Stock Data:")
    print(f" • File                 : {stock_file}")
    print(f" • Rows loaded          : {stock_rows if stock_rows>0 else 'N/A'}")
    print(f" • Date range           : {date_min} to {date_max}")
    if avg_vol is not None and not np.isnan(avg_vol):
        print(f" • Avg annualized vol   : {avg_vol:.2%}")
    else:
        print(f" • Avg annualized vol   : N/A (volatility column missing or insufficient history)")
    print()
    summary['stock'] = {
        'file': str(stock_file),
        'rows': stock_rows,
        'date_min': str(date_min),
        'date_max': str(date_max),
        'avg_annual_vol': float(avg_vol) if avg_vol is not None and not np.isnan(avg_vol) else None
    }

# ---- Risk-free rate ----
rf_file = find_first_existing(RISK_FREE_PATHS)
if rf_file is None:
    print("Risk-Free Rates: NOT FOUND. (Expected RBI 91-day T-bill data in data/reference/ or /mnt/data/.)")
    summary['risk_free'] = None
else:
    rf = _load_csv(rf_file)
    # normalize typical column names
    rf_date_col = next((c for c in rf.columns if c in ('date','day','timestamp')), None)
    rf_rate_col = next((c for c in rf.columns if c in ('rate_annual','rate','rate_pct','rate_91d','rate_91_day')), None)
    if rf_date_col:
        rf['date'] = pd.to_datetime(rf[rf_date_col], errors='coerce')
    if rf_rate_col:
        rf['rate_annual'] = pd.to_numeric(rf[rf_rate_col], errors='coerce')
    # compute simple summary
    rf_rows = len(rf)
    rf_avg = rf['rate_annual'].dropna().mean() if 'rate_annual' in rf.columns else None
    print("Risk-Free Rates:")
    print(f" • File                 : {rf_file}")
    print(f" • Observations         : {rf_rows if rf_rows>0 else 'N/A'}")
    if rf_avg is not None and not np.isnan(rf_avg):
        print(f" • Avg annual RF        : {rf_avg:.3%}")
    else:
        print(" • Avg annual RF        : N/A (rate column missing)")
    print()
    summary['risk_free'] = {
        'file': str(rf_file),
        'observations': rf_rows,
        'avg_annual_rate': float(rf_avg) if rf_avg is not None and not np.isnan(rf_avg) else None
    }

# ---- Dividend data ----
div_file = find_first_existing(DIVIDEND_PATHS)
if div_file is None:
    print("Dividend Data:")
    print(" • File                 : NOT FOUND (optional but recommended). Place dividend CSV in data/reference/")
    print()
    summary['dividends'] = None
else:
    div = _load_csv(div_file)
    # normalize possible column names
    date_cand = next((c for c in div.columns if 'date' in c), None)
    amt_cand = next((c for c in div.columns if any(x in c for x in ('amount','dividend','value'))), None)
    if date_cand:
        div['ex_date'] = pd.to_datetime(div[date_cand], errors='coerce')
    if amt_cand:
        div['dividend_amount'] = pd.to_numeric(div[amt_cand], errors='coerce')
    print("Dividend Data:")
    print(f" • File                 : {div_file}")
    print(f" • Dividend events      : {len(div) if len(div)>0 else 'N/A (empty)'}")
    if 'dividend_amount' in div.columns:
        total_div = div['dividend_amount'].dropna().sum()
        print(f" • Total dividend amt   : ₹{total_div:.2f}")
    else:
        print(" • dividend_amount column missing (please include 'dividend_amount' or similar).")
    print()
    summary['dividends'] = {'file': str(div_file), 'events': len(div)}

# ---- Option chain data ----
opt_file = find_first_existing(OPTION_CHAIN_PATHS)
if opt_file is None:
    print("Option Chain Data: NOT FOUND. Upload or place processed option chain under:")
    print("  ", "\n  ".join(str(p) for p in OPTION_CHAIN_PATHS))
    summary['option_chain'] = None
else:
    opt = _load_csv(opt_file)
    # ensure date / expiry parsed
    for dcol in ['date','expiry']:
        if dcol in opt.columns:
            opt[dcol] = pd.to_datetime(opt[dcol], errors='coerce')
    # try to detect mid columns
    call_mid_col = next((c for c in opt.columns if c in ('call_mid','call_price','call_mid_price')), None)
    put_mid_col = next((c for c in opt.columns if c in ('put_mid','put_price','put_mid_price')), None)
    oi_col = next((c for c in opt.columns if c in ('open_interest','oi')), None)
    rows = len(opt)
    n_dates = int(opt['date'].nunique()) if 'date' in opt.columns else "N/A"
    n_expiries = int(opt['expiry'].nunique()) if 'expiry' in opt.columns else "N/A"
    # ranges
    call_range = (opt[call_mid_col].min(), opt[call_mid_col].max()) if call_mid_col and call_mid_col in opt.columns else (None, None)
    put_range = (opt[put_mid_col].min(), opt[put_mid_col].max()) if put_mid_col and put_mid_col in opt.columns else (None, None)
    print("Option Chain Data:")
    print(f" • File                 : {opt_file}")
    print(f" • Option rows loaded   : {rows:,}")
    print(f" • Unique trade dates   : {n_dates}")
    print(f" • Unique expiries      : {n_expiries}")
    if call_range[0] is not None:
        print(f" • call_mid range       : ₹{call_range[0]:.3f} → ₹{call_range[1]:.2f}")
    else:
        print(" • call_mid range       : N/A (call_mid missing)")
    if put_range[0] is not None:
        print(f" • put_mid range        : ₹{put_range[0]:.3f} → ₹{put_range[1]:.2f}")
    else:
        print(" • put_mid range        : N/A (put_mid missing)")
    # liquidity checks
    if oi_col and oi_col in opt.columns:
        low_oi_pct = (opt[opt[oi_col] < 100].shape[0] / max(1, rows)) * 100
        print(f" • % rows with OI < 100 : {low_oi_pct:.2f}%")
    print()
    summary['option_chain'] = {
        'file': str(opt_file),
        'rows': rows,
        'unique_dates': n_dates,
        'unique_expiries': n_expiries,
        'call_mid_col': call_mid_col,
        'put_mid_col': put_mid_col
    }

# ---- Final summary object for programmatic use ----
print("=======================================================")
print("      DATA PREPARATION PIPELINE SUMMARY (programmatic)   ")
print("=======================================================\n")
for k,v in summary.items():
    print(f"{k.upper():<12} : {('FOUND' if v else 'MISSING')}")
print("\nYou can inspect the `summary` dict returned by this cell for programmatic access.")
# expose summary for downstream cells
_summary = summary


Stock Data:
 • File                 : ../data/processed/hdfc_stock_processed_271P.csv
 • Rows loaded          : 508
 • Date range           : 2023-11-01 to 2025-11-20
 • Avg annualized vol   : 18.68%

Risk-Free Rates:
 • File                 : ../data/processed/risk_free_rates_271P.csv
 • Observations         : 752
 • Avg annual RF        : 6.708%

Dividend Data:
 • File                 : ../data/processed/hdfc_dividends_271P.csv
 • Dividend events      : 3
 • Total dividend amt   : ₹23.25

Option Chain Data:
 • File                 : ../data/raw/hdfc_option_chain_raw.csv
 • Option rows loaded   : 8,694
 • Unique trade dates   : 478
 • Unique expiries      : 102
 • call_mid range       : ₹0.005 → ₹115.75
 • put_mid range        : ₹0.000 → ₹94.11
 • % rows with OI < 100 : 0.00%

      DATA PREPARATION PIPELINE SUMMARY (programmatic)   

STOCK        : FOUND
RISK_FREE    : FOUND
DIVIDENDS    : FOUND
OPTION_CHAIN : FOUND

You can inspect the `summary` dict returned by this cell for progra

In [45]:
import pandas as pd
import numpy as np
from pathlib import Path
import yaml

# Load configuration
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("ATM Lookup Table Generation\n")

# Load stock data
stock_data = pd.read_csv('../data/processed/hdfc_stock_processed_271P_enriched.csv')
stock_data['date'] = pd.to_datetime(stock_data['date']).dt.tz_localize(None)

print(f"Stock data loaded: {len(stock_data)} rows")

# Load option chain (test multiple paths)
option_chain_paths = [
    '../data/raw/hdfc_option_chain_raw.csv',
    '../data/processed/option_chain_synthetic_271P.csv',
    '/mnt/data/option_chain_synthetic_0271.csv'
]

option_chain = None

for path in option_chain_paths:
    try:
        option_chain = pd.read_csv(path)
        print(f"Option chain loaded from: {path}")
        break
    except FileNotFoundError:
        continue

if option_chain is None:
    raise FileNotFoundError("Option chain file not found.")

option_chain['date'] = pd.to_datetime(option_chain['date']).dt.tz_localize(None)
option_chain['expiry'] = pd.to_datetime(option_chain['expiry']).dt.tz_localize(None)

print(f"Option chain rows: {len(option_chain)}\n")

# Extract parameters from config
target_dte = config['trading']['target_expiry_days']
dte_tolerance = config['trading']['expiry_tolerance_days']
max_spread_pct = config['liquidity']['max_bid_ask_spread_pct']
min_oi = config['liquidity']['min_open_interest']
min_volume = config['liquidity']['min_daily_volume']
min_premium = config['liquidity']['min_option_premium']

atm_records = []
skipped_dates_no_options = 0
total_dates = len(stock_data)

# ATM selection loop
for idx, stock_row in stock_data.iterrows():
    trade_date = stock_row['date']
    stock_price = stock_row['adj_close']

    day_options = option_chain[option_chain['date'] == trade_date].copy()

    if day_options.empty:
        skipped_dates_no_options += 1
        continue

    if 'days_to_expiry' not in day_options.columns:
        day_options['days_to_expiry'] = (day_options['expiry'] - trade_date).dt.days

    dte_filtered = day_options[
        (day_options['days_to_expiry'] >= target_dte - dte_tolerance) &
        (day_options['days_to_expiry'] <= target_dte + dte_tolerance)
    ]

    if dte_filtered.empty:
        skipped_dates_no_options += 1
        continue

    dte_filtered = dte_filtered.copy()
    dte_filtered['strike_distance'] = (dte_filtered['strike'] - stock_price).abs()
    atm_strike = dte_filtered.loc[dte_filtered['strike_distance'].idxmin(), 'strike']

    atm_candidates = dte_filtered[dte_filtered['strike'] == atm_strike].copy()

    atm_candidates['dte_distance'] = (
        atm_candidates['days_to_expiry'] - target_dte
    ).abs()

    best_row = atm_candidates.loc[atm_candidates['dte_distance'].idxmin()]

    skip_reasons = []

    if best_row.get('open_interest', 0) < min_oi:
        skip_reasons.append(f"Low OI ({best_row.get('open_interest', 0)})")

    if best_row.get('volume', 0) < min_volume:
        skip_reasons.append(f"Low volume ({best_row.get('volume', 0)})")

    if best_row.get('call_spread_pct', 0) > max_spread_pct:
        skip_reasons.append(f"Wide call spread ({best_row.get('call_spread_pct', 0):.2f}%)")

    if best_row.get('put_spread_pct', 0) > max_spread_pct:
        skip_reasons.append(f"Wide put spread ({best_row.get('put_spread_pct', 0):.2f}%)")

    if best_row.get('call_mid', 0) < min_premium:
        skip_reasons.append(f"Low call premium (₹{best_row.get('call_mid', 0):.2f})")

    if best_row.get('put_mid', 0) < min_premium:
        skip_reasons.append(f"Low put premium (₹{best_row.get('put_mid', 0):.2f})")

    is_liquid = len(skip_reasons) == 0
    skip_reason = "; ".join(skip_reasons)

    record = {
        'trade_date': trade_date,
        'expiry_date': best_row['expiry'],
        'days_to_expiry': best_row['days_to_expiry'],
        'stock_price': stock_price,
        'atm_strike': atm_strike,
        'moneyness': stock_price / atm_strike,
        'call_bid': best_row.get('call_bid', np.nan),
        'call_ask': best_row.get('call_ask', np.nan),
        'call_mid': best_row.get('call_mid', np.nan),
        'put_bid': best_row.get('put_bid', np.nan),
        'put_ask': best_row.get('put_ask', np.nan),
        'put_mid': best_row.get('put_mid', np.nan),
        'implied_vol': best_row.get('implied_vol', np.nan),
        'open_interest': best_row.get('open_interest', 0),
        'volume': best_row.get('volume', 0),
        'call_spread_pct': best_row.get('call_spread_pct', np.nan),
        'put_spread_pct': best_row.get('put_spread_pct', np.nan),
        'risk_free_rate': best_row.get('risk_free_rate', 0.0668),
        'liquid': is_liquid,
        'skip_reason': skip_reason
    }

    atm_records.append(record)

# Convert to DataFrame
if len(atm_records) == 0:
    raise ValueError("No ATM records generated. Check date alignment or option chain content.")

atm_lookup = pd.DataFrame(atm_records)

print("ATM Lookup Table Summary:")
print(f"Total records: {len(atm_lookup)}")
print(f"Liquid trades: {atm_lookup['liquid'].sum()}")
print(f"Skipped (liquidity): {len(atm_lookup) - atm_lookup['liquid'].sum()}")
print(f"Dates with no options: {skipped_dates_no_options}\n")

output_main = Path('../data/processed/atm_lookup_271P.csv')
atm_lookup.to_csv(output_main, index=False)
print(f"Saved: {output_main}")

output_liquid = Path('../data/processed/atm_lookup_liquid_271P.csv')
atm_lookup[atm_lookup['liquid']].to_csv(output_liquid, index=False)
print(f"Saved liquid-only file: {output_liquid}")


ATM Lookup Table Generation

Stock data loaded: 508 rows
Option chain loaded from: ../data/raw/hdfc_option_chain_raw.csv
Option chain rows: 8694

ATM Lookup Table Summary:
Total records: 478
Liquid trades: 478
Skipped (liquidity): 0
Dates with no options: 30

Saved: ../data/processed/atm_lookup_271P.csv
Saved liquid-only file: ../data/processed/atm_lookup_liquid_271P.csv


In [46]:
common = set(stock_data['date']) & set(option_chain['date'])
print("Common trading dates:", len(common))

Common trading dates: 478


In [47]:
atm_lookup.columns


Index(['trade_date', 'expiry_date', 'days_to_expiry', 'stock_price',
       'atm_strike', 'moneyness', 'call_bid', 'call_ask', 'call_mid',
       'put_bid', 'put_ask', 'put_mid', 'implied_vol', 'open_interest',
       'volume', 'call_spread_pct', 'put_spread_pct', 'risk_free_rate',
       'liquid', 'skip_reason'],
      dtype='object')

In [48]:
len(atm_lookup)

478

In [49]:
from src.pricing_models.black_scholes import BlackScholesModel

bsm = BlackScholesModel()

synthetic_prices = []

for _, row in atm_lookup.iterrows():
    S = row["stock_price"]
    K = row["atm_strike"]
    T = row["days_to_expiry"] / 365
    r = row["risk_free_rate"]
    sigma = row["implied_vol"]

    price = bsm.price_call(
        stock_price=S,
        strike_price=K,
        time_to_expiry=T,
        risk_free_rate=r,
        volatility=sigma
    )
    synthetic_prices.append(price)

atm_lookup["synthetic_price"] = synthetic_prices

atm_lookup.to_csv("../data/processed/atm_lookup_271P.csv", index=False)

print("Synthetic prices added. ATM lookup updated and saved.")


Synthetic prices added. ATM lookup updated and saved.


In [52]:
# === Universal PDF Export Cell (Jupyter, macOS, LaTeX-free, automatic install) ===
import os, sys, subprocess, importlib
from nbconvert import HTMLExporter

# ---- Step 1: Install Playwright if missing ----
def ensure_playwright():
    spec = importlib.util.find_spec("playwright")
    if spec is None:
        print("Playwright not found. Installing into this kernel environment...")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "playwright", "nbconvert"], check=True)
        print("Installing Chromium browser (required for PDF generation)...")
        subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=True)
        print("Playwright + Chromium installed.")
    else:
        print("Playwright already installed.")

ensure_playwright()

# After install, import async API
from playwright.async_api import async_playwright

# ---- Step 2: Ask for notebook filename ----
notebook_name = input("Enter notebook name (e.g. 01_data_acquisition.ipynb): ").strip()
if not notebook_name:
    raise SystemExit("Notebook name required.")
if not os.path.exists(notebook_name):
    raise FileNotFoundError(f"Notebook not found: {notebook_name}")

html_name = notebook_name.replace(".ipynb", ".html")
pdf_name = notebook_name.replace(".ipynb", ".pdf")

# ---- Step 3: Export notebook → HTML ----
print(f"Exporting {notebook_name} → {html_name} ...")
html_exporter = HTMLExporter()
html_exporter.exclude_input = False
html_exporter.template_name = "classic"

body, resources = html_exporter.from_filename(notebook_name)
with open(html_name, "w", encoding="utf-8") as f:
    f.write(body)
print(f"HTML saved as {html_name}")

# ---- Step 4: Render HTML → PDF using Chromium headless ----
abs_html = os.path.abspath(html_name)
file_url = "file://" + abs_html
print("Rendering PDF using Chromium...")

async def _export_pdf():
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        ctx = await browser.new_context()
        page = await ctx.new_page()
        await page.goto(file_url, wait_until="networkidle")
        await page.pdf(
            path=pdf_name,
            format="A4",
            print_background=True,
            margin={"top": "12mm", "bottom": "12mm", "left": "12mm", "right": "12mm"}
        )
        await browser.close()

await _export_pdf()

print(f"PDF successfully created: {pdf_name}")


Playwright already installed.


Enter notebook name (e.g. 01_data_acquisition.ipynb):  02_data_processing_and_etl.ipynb


Exporting 02_data_processing_and_etl.ipynb → 02_data_processing_and_etl.html ...
HTML saved as 02_data_processing_and_etl.html
Rendering PDF using Chromium...
PDF successfully created: 02_data_processing_and_etl.pdf


In [53]:
atm_lookup.head()

,trade_date,expiry_date,days_to_expiry,stock_price,atm_strike,moneyness,call_bid,call_ask,call_mid,put_bid,...,put_mid,implied_vol,open_interest,volume,call_spread_pct,put_spread_pct,risk_free_rate,liquid,skip_reason,synthetic_price
0,2023-12-15,2024-01-18,34,806.219299,810,0.995332,13.75,13.95,13.870425,12.45,...,12.536601,0.134515,5948,150,1.441917,1.595329,0.068,True,,13.870425
1,2023-12-18,2024-01-18,31,805.805542,810,0.994822,12.80,12.95,12.875233,12.30,...,12.405147,0.134954,5110,384,1.165027,1.612234,0.068,True,,12.875233
2,2023-12-19,2024-01-18,30,804.442871,800,1.005554,17.25,17.50,17.384244,8.35,...,8.482612,0.135855,5910,506,1.438084,2.947206,0.068,True,,17.384244
3,2023-12-20,2024-01-25,36,806.438232,810,0.995603,14.50,14.70,14.615347,12.65,...,12.762743,0.135452,5290,500,1.368425,1.567061,0.068,True,,14.615347
4,2023-12-21,2024-01-25,35,820.892822,820,1.001089,17.25,17.50,17.393614,11.10,...,11.171337,0.139222,5763,246,1.437309,1.342722,0.068,True,,17.393614


In [ ]:
# Cell: recompute synthetic using realized volatility (non-peeking)
import sys, os
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
from src.pricing_models.black_scholes import BlackScholesModel

atm = pd.read_csv("../data/processed/atm_lookup_271P.csv")
# ensure stock_data has realized vol column if used; otherwise compute a short rolling vol
try:
    stock = pd.read_csv("../data/processed/hdfc_stock_processed_271P_enriched.csv")
    if "realized_vol_annual" in stock.columns:
        # merge realized vol by trade_date
        stock["trade_date"] = pd.to_datetime(stock["date"])
        atm["trade_date"] = pd.to_datetime(atm["trade_date"])
        atm = atm.merge(stock[["trade_date", "realized_vol_annual"]], on="trade_date", how="left")
        atm["use_vol"] = atm["realized_vol_annual"].fillna(atm["implied_vol"])
    else:
        # fallback: compute 30-day rolling daily vol annualized from close returns
        stock["date"] = pd.to_datetime(stock["date"])
        stock = stock.sort_values("date")
        stock["ret"] = stock["adj_close"].pct_change()
        stock["rolling30vol"] = stock["ret"].rolling(30).std() * (252**0.5)
        stock["trade_date"] = stock["date"]
        atm = atm.merge(stock[["trade_date", "rolling30vol"]], on="trade_date", how="left")
        atm["use_vol"] = atm["rolling30vol"].fillna(atm["implied_vol"])
except Exception:
    atm["use_vol"] = atm["implied_vol"]

bsm = BlackScholesModel()
synthetic = []
for _, r in atm.iterrows():
    S = float(r["stock_price"])
    K = float(r["atm_strike"])
    T = max(float(r["days_to_expiry"]) / 365.0, 1e-8)
    rf = float(r["risk_free_rate"])
    sigma = float(r["use_vol"])
    # try the typical API, fallback to positional
    try:
        p = bsm.price_call(stock_price=S, strike_price=K, time_to_expiry=T, risk_free_rate=rf, implied_vol=sigma)
    except TypeError:
        p = bsm.price_call(S, K, T, rf, sigma)
    synthetic.append(float(p))

atm["synthetic_price"] = synthetic
atm.to_csv("../data/processed/atm_lookup_271P.csv", index=False)
print("Recomputed synthetic using realized/rolling vol. Saved updated atm_lookup.")
